In [ ]:
from pathlib import Path
import io, gzip, unicodedata
import os, sys, platform, random
import numpy as np
import regex as re
from xml.etree import ElementTree as ET

import torch
from datasets import Dataset
import evaluate
import inspect
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)

#  Run knobs 
SEED = 42
PROJECT_DIR = "./nllb_bilingual_en_es_pt"
ROOT = Path(r"J:\FINAL PROJECT")
TMX_ES = ROOT / "data" / "en-es.tmx"
TMX_PT = ROOT / "data" / "en-pt.tmx"

# dataset sizes
MAX_SAMPLES = 1200  # max per pair (train pool)
EVAL_SAMPLES = 300  # max per pair (eval pool)

# tokenization / model
MAX_SRC_LEN = 64
MAX_TGT_LEN = 64

# training
BATCH = 8
GRAD_ACCUM = 1
MAX_STEPS = 100
LR = 3e-5

# language codes for NLLB
NLLB_CKPT = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
ES_LANG = "spa_Latn"
PT_LANG = "por_Latn"

# fast-eval presets
FAST_MAX_NEW = 96
FAST_MIN_NEW = 2
FAST_BEAMS = 1
FAST_BATCH = 128

# Repro & device
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
use_fp16 = torch.cuda.is_available() and not use_bf16
os.makedirs(PROJECT_DIR, exist_ok=True)


In [2]:
def _nfc_ws(s: str) -> str:
    s = unicodedata.normalize("NFC", s or "")
    s = s.replace("\u00A0", " ").replace("\u202F", " ")
    return re.sub(r"\s+", " ", s).strip()

def _norm_lang(tag: str) -> str:
    if not tag: return ""
    t = tag.replace("_", "-").lower()
    if t.startswith("eng"): return "en"
    if t.startswith("spa"): return "es"
    if t.startswith("por"): return "pt"
    return t.split("-")[0]

def _open_maybe_gz(path: str):
    return io.TextIOWrapper(gzip.open(path, "rb"), encoding="utf-8", errors="ignore") if path.endswith(".gz") \
           else io.open(path, "r", encoding="utf-8", errors="ignore")

def load_tmx_as_dataset(tmx_path: str, src="en", tgt="es", max_samples=None, seed=42) -> Dataset:
    p = Path(tmx_path); assert p.exists(), f"Missing TMX: {tmx_path}"
    src = _norm_lang(src); tgt = _norm_lang(tgt)
    pairs = []
    with _open_maybe_gz(str(p)) as fh:
        context = ET.iterparse(fh, events=("start", "end"))
        _, root = next(context)
        cur_src = cur_tgt = None
        for event, elem in context:
            tag = elem.tag.lower()
            if event == "start" and tag.endswith("tu"):
                cur_src = cur_tgt = None
            if event == "end" and tag.endswith("tuv"):
                lang = _norm_lang(elem.attrib.get("{http://www.w3.org/XML/1998/namespace}lang",
                                                  elem.attrib.get("lang","")))
                seg = next((c for c in elem if c.tag.lower().endswith("seg")), None)
                if seg is not None:
                    text = _nfc_ws("".join(seg.itertext()))
                    if lang == src: cur_src = text
                    elif lang == tgt: cur_tgt = text
            if event == "end" and tag.endswith("tu"):
                if cur_src and cur_tgt:
                    pairs.append((cur_src, cur_tgt))
                root.clear()
                if max_samples and len(pairs) >= max_samples:
                    break

    if max_samples and len(pairs) > max_samples:
        rnd = random.Random(seed)
        idx = list(range(len(pairs))); rnd.shuffle(idx); idx = idx[:max_samples]
        pairs = [pairs[i] for i in idx]

    return Dataset.from_dict({
        "src_text": [s for s,_ in pairs],
        "tgt_text": [t for _,t in pairs]
    })

def light_clean(ds: Dataset, src_lang_check="en"):
    import langid
    langid.set_languages([src_lang_check])

    URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
    PUNCT_ONLY_RE = re.compile(r"^\p{P}+$", re.UNICODE)

    def clean_example(ex):
        s = URL_RE.sub("<URL>", _nfc_ws(ex["src_text"]))
        t = URL_RE.sub("<URL>", _nfc_ws(ex["tgt_text"]))
        return {"src_text": s, "tgt_text": t}

    def ok(ex):
        s,t = ex["src_text"], ex["tgt_text"]
        if len(s.split()) < 2 or len(t.split()) < 2: return False
        if PUNCT_ONLY_RE.match(s) or PUNCT_ONLY_RE.match(t): return False
        if len(s.split()) > 200 or len(t.split()) > 200: return False
        ls, cs = langid.classify(s)
        if ls != src_lang_check and cs > 0.7: return False
        return True

    def dedup(ds_):
        seen, keep = set(), []
        for i, ex in enumerate(ds_):
            k = (ex["src_text"].lower(), ex["tgt_text"].lower())
            if k not in seen:
                seen.add(k); keep.append(i)
        return ds_.select(keep)

    ds = ds.map(clean_example)
    ds = dedup(ds)
    ds = ds.filter(ok)
    return ds

# Load raw + clean
ds_es_all = light_clean(load_tmx_as_dataset(TMX_ES, "en", "es", MAX_SAMPLES+EVAL_SAMPLES, SEED))
ds_pt_all = light_clean(load_tmx_as_dataset(TMX_PT, "en", "pt", MAX_SAMPLES+EVAL_SAMPLES, SEED))

print("Clean sizes:", f"EN→ES {len(ds_es_all)} | EN→PT {len(ds_pt_all)}")


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1499 [00:00<?, ? examples/s]

Clean sizes: EN→ES 1498 | EN→PT 1492


In [3]:
def forced_bos_id(tok, lang_code: str) -> int:
    m = getattr(tok, "lang_code_to_id", None)
    if isinstance(m, dict) and lang_code in m:
        return m[lang_code]
    if hasattr(tok, "get_lang_id"):
        return tok.get_lang_id(lang_code)
    i = tok.convert_tokens_to_ids(lang_code)
    if i is None or i == tok.unk_token_id:
        raise RuntimeError(f"Cannot resolve BOS id for {lang_code}")
    return i

def build_nllb_pair(lang_code: str):
    tok = AutoTokenizer.from_pretrained(NLLB_CKPT, use_fast=True)
    tok.src_lang = SRC_LANG
    tok.tgt_lang = lang_code

    model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CKPT)
    lang_id = forced_bos_id(tok, lang_code)

    for cfg in (model.config, model.generation_config):
        cfg.forced_bos_token_id = lang_id
        cfg.decoder_start_token_id = lang_id

    if device == "cuda":
        model.to(device)
    return tok, model

tok_es, mod_es = build_nllb_pair(ES_LANG)
tok_pt, mod_pt = build_nllb_pair(PT_LANG)


In [ ]:
# #bleualign-like alignment
# SIM_MODE = "char3-jaccard"   # "chrf" or "char3-jaccard"
# ALLOW_MERGE = False          # True enables 1-2 / 2-1 (slower)
# BLEUALIGN_MAX_JUMP = 3       # tighter band = faster
# BLEUALIGN_MIN_SIM = 0.28     # raise a bit if using char3-jaccard (0.26–0.32)

# # ----- fast similarity helpers -----
# def _char_ngrams(s, n=3):
#     s = _norm(s)
#     if len(s) < n: return set([s])
#     return {s[i:i+n] for i in range(len(s)-n+1)}

# def _sim_char3_jaccard(a, b):
#     A = _char_ngrams(a, 3); B = _char_ngrams(b, 3)
#     if not A and not B: return 1.0
#     inter = len(A & B); uni = len(A | B)
#     return inter / max(1, uni)

# def _chrF(a: str, b: str) -> float:
#     if SIM_MODE == "char3-jaccard":
#         return _sim_char3_jaccard(a, b)
#     else:
#         # original chrF via sacrebleu
#         return sacrebleu.sentence_chrf(_norm(a), [_norm(b)]).score / 100.0

# # ----- banded, cached DP (on-demand sims) -----
# def _align_dp(src_mt: List[str], tgt: List[str]) -> List[Tuple[List[int], List[int], float]]:
#     n, m = len(src_mt), len(tgt)
#     NEG_INF = -1e9
#     est_slope = (m+1) / max(1, n+1)

#     dp = np.full((n+1, m+1), NEG_INF, dtype=np.float32)
#     bp = np.empty((n+1, m+1), dtype=object)
#     dp[0,0] = 0.0; bp[0,0] = None

#     # sim caches (compute on demand)
#     sim11 = {}
#     sim12 = {}
#     sim21 = {}

#     def s11(i,j):
#         k = (i,j)
#         if k not in sim11:
#             sim11[k] = _chrF(src_mt[i], tgt[j])
#         return sim11[k]

#     def s12(i,j):
#         # i with j+j+1
#         k = (i,j)
#         if k not in sim12:
#             sim12[k] = _chrF(src_mt[i], tgt[j] + " " + tgt[j+1]) + BLEUALIGN_MERGE_BONUS
#         return sim12[k]

#     def s21(i,j):
#         # i+i+1 with j
#         k = (i,j)
#         if k not in sim21:
#             sim21[k] = _chrF(src_mt[i] + " " + src_mt[i+1], tgt[j]) + BLEUALIGN_MERGE_BONUS
#         return sim21[k]

#     for i in range(n+1):
#         j_center = int(round(i * est_slope))
#         for j in _band_indices(i, m+1, j_center, BLEUALIGN_MAX_JUMP):
#             if i==0 and j==0:
#                 continue
#             best_score, best_bp = NEG_INF, None

#             # gap in T (consume S)
#             if i>0:
#                 sc = dp[i-1, j] - BLEUALIGN_GAP_PEN
#                 if sc > best_score:
#                     best_score = sc; best_bp = (i-1, j, [i-1], [])

#             # gap in S (consume T)
#             if j>0:
#                 sc = dp[i, j-1] - BLEUALIGN_GAP_PEN
#                 if sc > best_score:
#                     best_score = sc; best_bp = (i, j-1, [], [j-1])

#             # 1-1
#             if i>0 and j>0:
#                 sc = dp[i-1, j-1] + s11(i-1, j-1)
#                 if sc > best_score:
#                     best_score = sc; best_bp = (i-1, j-1, [i-1], [j-1])

#             if ALLOW_MERGE:
#                 # 1-2
#                 if i>0 and j>1:
#                     sc = dp[i-1, j-2] + s12(i-1, j-2)
#                     if sc > best_score:
#                         best_score = sc; best_bp = (i-1, j-2, [i-1], [j-2, j-1])
#                 # 2-1
#                 if i>1 and j>0:
#                     sc = dp[i-2, j-1] + s21(i-2, j-1)
#                     if sc > best_score:
#                         best_score = sc; best_bp = (i-2, j-1, [i-2, i-1], [j-1])

#             dp[i, j] = best_score
#             bp[i, j] = best_bp

#     # backtrack
#     i, j = n, m
#     chunks = []
#     while not (i==0 and j==0):
#         prev = bp[i, j]
#         if prev is None: break
#         pi, pj, si, tj = prev
#         if si or tj:
#             # compute realized sim for reporting
#             s_join = " ".join(src_mt[k] for k in si) if si else ""
#             t_join = " ".join(tgt[k] for k in tj) if tj else ""
#             sim = _chrF(s_join, t_join) if (s_join and t_join) else 0.0
#             chunks.append((si, tj, sim))
#         i, j = pi, pj
#     chunks.reverse()
#     return chunks


In [ ]:
# print("Running BleuAlign for EN→ES…")
# ds_es_aligned = bleualign_align(ds_es_all, mod_es, tok_es, "en-es", ES_LANG)

# print("Running BleuAlign for EN→PT…")
# ds_pt_aligned = bleualign_align(ds_pt_all, mod_pt, tok_pt, "en-pt", PT_LANG)

# print("Aligned sizes:", f"EN→ES {len(ds_es_aligned)} | EN→PT {len(ds_pt_aligned)}")


Running BleuAlign for EN→ES…
[en-es] MT 1498 src sentences to spa_Latn…


KeyboardInterrupt: 

In [ ]:
es_eval = min(EVAL_SAMPLES, max(50, int(0.10 * len(ds_es_aligned))))
pt_eval = min(EVAL_SAMPLES, max(50, int(0.10 * len(ds_pt_aligned))))

split_es = ds_es_aligned.train_test_split(test_size=es_eval, seed=SEED, shuffle=True)
split_pt = ds_pt_aligned.train_test_split(test_size=pt_eval, seed=SEED, shuffle=True)

train_es, valid_es = split_es["train"], split_es["test"]
train_pt, valid_pt = split_pt["train"], split_pt["test"]

print("Final split sizes (BLEUALIGN):",
      f"EN→ES train {len(train_es)} / valid {len(valid_es)}  |  EN→PT train {len(train_pt)} / valid {len(valid_pt)}")


In [ ]:
def preprocess(ds, tok):
    def _fn(batch):
        X = tok(batch["src_text"], truncation=True, max_length=MAX_SRC_LEN)
        try:
            Y = tok(text_target=batch["tgt_text"], truncation=True, max_length=MAX_TGT_LEN)
        except TypeError:
            from contextlib import contextmanager
            cm = tok.as_target_tokenizer() if hasattr(tok, "as_target_tokenizer") else contextmanager(lambda: (yield))()
            with cm:
                Y = tok(batch["tgt_text"], truncation=True, max_length=MAX_TGT_LEN)
        X["labels"] = Y["input_ids"]
        return X
    return ds.map(_fn, batched=True, remove_columns=ds.column_names)

train_es_tok = preprocess(train_es, tok_es)
valid_es_tok = preprocess(valid_es, tok_es)
train_pt_tok = preprocess(train_pt, tok_pt)
valid_pt_tok = preprocess(valid_pt, tok_pt)


In [ ]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def _autocast_if_cuda():
    if torch.cuda.is_available():
        return torch.autocast("cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16)
    return torch.cuda.amp.autocast(enabled=False)

@torch.inference_mode()
def batch_generate_fast(model, tok, texts,
                        max_len_src=MAX_SRC_LEN, max_new=FAST_MAX_NEW,
                        beams=FAST_BEAMS, batch_size=FAST_BATCH,
                        pad_to_multiple_of=8):

    enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
              max_length=max_len_src, pad_to_multiple_of=pad_to_multiple_of)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    lang_id = forced_bos_id(tok, tok.tgt_lang)

    preds = []
    model.eval()
    if getattr(model.config, "use_cache", True) is False:
        model.config.use_cache = True

    with _autocast_if_cuda():
        for i in range(0, enc["input_ids"].size(0), batch_size):
            sl = slice(i, i+batch_size)
            out = model.generate(
                input_ids=enc["input_ids"][sl],
                attention_mask=enc.get("attention_mask", None)[sl] if enc.get("attention_mask") is not None else None,
                max_new_tokens=max_new,
                min_new_tokens=FAST_MIN_NEW,
                num_beams=beams,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
                do_sample=False,
                forced_bos_token_id=lang_id,
                decoder_start_token_id=lang_id,
                use_cache=True,
            )
            preds.extend(tok.batch_decode(out, skip_special_tokens=True))

    return [p.strip() for p in preds]

def evaluate_pair_fast(model, tok, valid_ds, name, limit=128):
    src = [r["src_text"] for r in valid_ds][:limit]
    refs = [r["tgt_text"] for r in valid_ds][:limit]
    preds = batch_generate_fast(model, tok, src)
    refs_bleu = [[r] for r in refs]

    bleu = bleu_metric.compute(predictions=preds, references=refs_bleu)["score"]
    chrf = chrf_metric.compute(predictions=preds, references=refs_bleu)["score"]
    exact = float(np.mean([int(p == r) for p, r in zip(preds, refs)]) * 100.0)

    print(f"{name}: BLEU {bleu:.2f} | chrF {chrf:.2f} | exact {exact:.2f}% on {len(preds)}")
    return {"bleu": bleu, "chrf": chrf, "exact": exact}

print("🔎 EN→ES zero-shot:")
evaluate_pair_fast(mod_es, tok_es, valid_es, "EN→ES zero-shot")

print("🔎 EN→PT zero-shot:")
evaluate_pair_fast(mod_pt, tok_pt, valid_pt, "EN→PT zero-shot")


In [ ]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def _autocast_if_cuda():
    if torch.cuda.is_available():
        return torch.autocast("cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16)
    return torch.cuda.amp.autocast(enabled=False)

@torch.inference_mode()
def batch_generate_fast(model, tok, texts,
                        max_len_src=MAX_SRC_LEN, max_new=FAST_MAX_NEW,
                        beams=FAST_BEAMS, batch_size=FAST_BATCH,
                        pad_to_multiple_of=8):

    enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
              max_length=max_len_src, pad_to_multiple_of=pad_to_multiple_of)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    lang_id = forced_bos_id(tok, tok.tgt_lang)

    preds = []
    model.eval()
    if getattr(model.config, "use_cache", True) is False:
        model.config.use_cache = True

    with _autocast_if_cuda():
        for i in range(0, enc["input_ids"].size(0), batch_size):
            sl = slice(i, i+batch_size)
            out = model.generate(
                input_ids=enc["input_ids"][sl],
                attention_mask=enc.get("attention_mask", None)[sl] if enc.get("attention_mask") is not None else None,
                max_new_tokens=max_new,
                min_new_tokens=FAST_MIN_NEW,
                num_beams=beams,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
                do_sample=False,
                forced_bos_token_id=lang_id,
                decoder_start_token_id=lang_id,
                use_cache=True,
            )
            preds.extend(tok.batch_decode(out, skip_special_tokens=True))

    return [p.strip() for p in preds]

def evaluate_pair_fast(model, tok, valid_ds, name, limit=128):
    src = [r["src_text"] for r in valid_ds][:limit]
    refs = [r["tgt_text"] for r in valid_ds][:limit]
    preds = batch_generate_fast(model, tok, src)
    refs_bleu = [[r] for r in refs]

    bleu = bleu_metric.compute(predictions=preds, references=refs_bleu)["score"]
    chrf = chrf_metric.compute(predictions=preds, references=refs_bleu)["score"]
    exact = float(np.mean([int(p == r) for p, r in zip(preds, refs)]) * 100.0)

    print(f"{name}: BLEU {bleu:.2f} | chrF {chrf:.2f} | exact {exact:.2f}% on {len(preds)}")
    return {"bleu": bleu, "chrf": chrf, "exact": exact}

print("🔎 EN→ES zero-shot:")
evaluate_pair_fast(mod_es, tok_es, valid_es, "EN→ES zero-shot")

print("🔎 EN→PT zero-shot:")
evaluate_pair_fast(mod_pt, tok_pt, valid_pt, "EN→PT zero-shot")


In [ ]:
def make_training_args(**wanted):
    accepted = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    filtered = {k: v for k, v in wanted.items() if k in accepted}
    dropped = sorted(set(wanted) - set(filtered))
    if dropped:
        print("Dropped unsupported keys:", dropped)
    return TrainingArguments(**filtered)

def train_one(model, tok, train_tok, out_dir):
    args = make_training_args(
        output_dir=out_dir,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_steps=MAX_STEPS,
        learning_rate=LR,
        lr_scheduler_type="linear",
        warmup_ratio=0.05,
        weight_decay=0.01,
        logging_steps=100,
        report_to="none",
        bf16=use_bf16,
        fp16=use_fp16,
        no_cuda=(device!="cuda"),
        dataloader_num_workers=0,
        dataloader_pin_memory=(device=="cuda"),
        evaluation_strategy="no",
        save_strategy="no",
        optim=("adamw_torch_fused" if device=="cuda" else "adamw_torch"),
        group_by_length=True,
    )

    collator = DataCollatorForSeq2Seq(tokenizer=tok, model=model, padding="longest")
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=None,
        data_collator=collator,
        tokenizer=tok,
    )
    trainer.train()

print("🚀 Training EN→ES …")
train_one(mod_es, tok_es, train_es_tok, out_dir=f"{PROJECT_DIR}/ckpt_es")

print("🚀 Training EN→PT …")
train_one(mod_pt, tok_pt, train_pt_tok, out_dir=f"{PROJECT_DIR}/ckpt_pt")


In [ ]:
print("✅ EN→ES finetuned:")
evaluate_pair_fast(mod_es, tok_es, valid_es, "EN→ES finetuned")

print("✅ EN→PT finetuned:")
evaluate_pair_fast(mod_pt, tok_pt, valid_pt, "EN→PT finetuned")

@torch.inference_mode()
def translate(model, tok, text, max_new_tokens=128, num_beams=4, tgt_lang=None):
    enc = tok([text], return_tensors="pt", padding=True, truncation=True, max_length=MAX_SRC_LEN)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    lang = tgt_lang or tok.tgt_lang
    lang_id = forced_bos_id(tok, lang)
    out = model.generate(
        **enc,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=3,
        early_stopping=True,
        forced_bos_token_id=lang_id,
        decoder_start_token_id=lang_id
    )
    return tok.batch_decode(out, skip_special_tokens=True)[0]

print("ES:", translate(mod_es, tok_es, "This is a small test of a machine translation system."))
print("PT:", translate(mod_pt, tok_pt, "How are you today?"))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import random, regex as re

def show_random_pairs(ds, n=10, seed=42):
    rng = random.Random(seed)
    idxs = rng.sample(range(len(ds)), min(n, len(ds)))
    for i in idxs:
        print(f"\n=== SAMPLE {i} ===")
        print("SRC:", ds[i]["src_text"])
        print("TGT:", ds[i]["tgt_text"])

def plot_length_ratios(ds, title="Length ratios (src/tgt)"):
    ratios = []
    for ex in ds:
        s = len(ex["src_text"].split())
        t = len(ex["tgt_text"].split())
        if t > 0:
            ratios.append(s/t)
    plt.figure()
    plt.hist(ratios, bins=50)
    plt.title(title)
    plt.xlabel("src_words / tgt_words")
    plt.ylabel("count")
    plt.show()
    return ratios

print("🔎 EN→ES random pairs:")
show_random_pairs(train_es, n=8)

print("\n📈 EN→ES length ratios (train):")
_ = plot_length_ratios(train_es, title="EN→ES train ratios")

print("\n🔎 EN→PT random pairs:")
show_random_pairs(train_pt, n=8)

print("\n📈 EN→PT length ratios (train):")
_ = plot_length_ratios(train_pt, title="EN→PT train ratios")
